In [ ]:
import os
import librosa
import numpy as np
import joblib
from music21 import note, stream


In [ ]:
model = joblib.load("rf_13_classifier.pkl")


In [ ]:
def create_audio_file(files):
    combined_audio = []
    sr = 16000
    silence = np.zeros(int(0.2 * sr))

    for i, file_name in enumerate(files):
        path = os.path.join("nsynth-train/audio", file_name)
        y, _ = librosa.load(path, sr=sr)
        combined_audio.append(y)

        if i < len(files) - 1:
            combined_audio.append(silence)

    return np.concatenate(combined_audio), sr


In [ ]:
def segment_audio_pyin(y, sr):
    f0, voiced_flag, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("A2"),
        fmax=librosa.note_to_hz("A5")
    )

    voiced = np.asarray(voiced_flag, dtype=bool)
    frame_samples = librosa.frames_to_samples(np.arange(len(voiced)))

    energy = librosa.feature.rms(y=y)[0]

    segments = []
    start = None

    for i in range(len(voiced)):

        is_voiced = voiced[i]
        low_energy = energy[i] < 0.01

        if is_voiced and start is None:
            start = frame_samples[i]

        elif (not is_voiced or low_energy) and start is not None:
            end = frame_samples[i]
            duration = (end - start) / sr
            if duration > 0.3:
                segments.append((int(start), int(end)))
            start = None

    if start is not None:
        duration = (len(y) - start) / sr
        if duration > 0.3:
            segments.append((int(start), len(y)))

    return segments


def merge_close_segments(segments, sr, gap_thresh=0.15):

    if len(segments) == 0:
        return []

    merged = []
    prev_start, prev_end = segments[0]

    for start, end in segments[1:]:

        gap = (start - prev_end) / sr

        if gap < gap_thresh:
            prev_end = end
        else:
            merged.append((prev_start, prev_end))
            prev_start, prev_end = start, end

    merged.append((prev_start, prev_end))
    return merged


In [ ]:
def extract_segment_features(segment_audio, sr):
    mfcc = np.mean(librosa.feature.mfcc(y=segment_audio, sr=sr, n_mfcc=13).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=segment_audio, sr=sr).T, axis=0)
    centroid = np.mean(librosa.feature.spectral_centroid(y=segment_audio, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(segment_audio))

    return np.hstack((mfcc, chroma, centroid, zcr))


In [ ]:
def predict_segment_pitch(segment_audio, sr):
    features = extract_segment_features(segment_audio, sr)
    features = features.reshape(1, -1)
    pitch = model.predict(features)[0]

    return int(pitch)


In [ ]:
def estimate_tempo_and_scale(notes, durations):
    note_durations = [d for n, d in zip(notes, durations) if n != "rest"]

    if len(note_durations) == 0:
        return 120, 1

    base = np.median(note_durations)

    candidates = [0.5, 1, 2, 4]

    best_tempo = None
    best_scale = None
    best_error = float("inf")

    for scale in candidates:

        tempo = 60 / (base / scale)

        beats = [d / (60 / tempo) for d in durations]

        error = np.mean([abs(b - round(b)) for b in beats])

        if error < best_error:
            best_error = error
            best_tempo = tempo
            best_scale = scale

    return best_tempo, best_scale


def quantize_duration(d):
    values = [0.25, 0.5, 1, 2, 3, 4]

    return min(values, key=lambda x: abs(x - d))


def process_durations(notes, durations):
    tempo, scale = estimate_tempo_and_scale(notes, durations)

    beats = [d / (60 / tempo) for d in durations]

    quantized = [quantize_duration(b) for b in beats]

    return tempo, scale, beats, quantized


def build_notes(y, sr, segments):
    notes = []
    durations = []

    for i in range(len(segments)):

        start, end = segments[i]
        segment_audio = y[start:end]

        if len(segment_audio) < 2000:
            continue

        if np.mean(np.abs(segment_audio)) < 0.01:
            continue

        # NOTE
        pitch = predict_segment_pitch(segment_audio, sr)
        duration = (end - start) / sr

        notes.append(pitch)
        durations.append(duration)

        # REST (between segments)
        if i < len(segments) - 1:
            next_start = segments[i+1][0]
            gap = (next_start - end) / sr

            if gap > 0.2:
                notes.append("rest")
                durations.append(gap)

    return notes, durations


In [ ]:
def create_seq_sheet(notes, durations, output_name="output"):
    score = stream.Stream()

    tempo, scale, beats, quantized = process_durations(notes, durations)

    for pitch, q in zip(notes, quantized):

        if pitch == "rest":
            n = note.Rest()
        else:
            n = note.Note()
            n.pitch.midi = int(pitch)

        n.quarterLength = q
        score.append(n)

    output_dir = os.path.dirname(output_name)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    score.write("musicxml", output_name + ".xml")
    print("Sheet music created at", output_name + ".xml")


In [ ]:
def run_pipeline(files, output_name):
    y, sr = create_audio_file(files)
    segments = segment_audio_pyin(y, sr)
    segments = merge_close_segments(segments, sr)
    notes, durations = build_notes(y, sr, segments)
    tempo, scale, beats, quantized = process_durations(notes, durations)

    create_seq_sheet(notes, durations, output_name)

    print("detected segments:", segments)
    print("predicted notes:", notes)
    print("notes with rests:", notes)
    print("durations:", durations)
    print("tempo:", tempo)
    print("scale chosen:", scale)
    print("raw durations:", durations)
    print("beats:", beats)
    print("quantized:", quantized)
    print("num segments:", len(segments))
    print("num notes:", len(notes))
    print("segment durations:", [(end-start)/sr for start,end in segments])
    print("total duration vs audio length:", sum(durations), len(y)/sr)

    return notes, durations, segments


In [ ]:
#Example
files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav"
]
notes, durations, segments = run_pipeline(files, "pitch_segmentation/hybrid_output")
